# Hierarchical reaction-diffusion

A cascade of two-field reaction-diffusion stages in which the *product* field of stage $k$ provides a spatial feed map for stage $k+1$. Two kinetic models are available:

- **Gray-Scott**: $\dot U = D_U \nabla^2 U - U V^2 + F(1-U)$, $\dot V = D_V \nabla^2 V + U V^2 - (F+k) V$.  The substrate $U$ is consumed by an autocatalytic conversion to $V$; $V$ is the natural "product".
- **FitzHugh-Nagumo**: $\dot U = a \nabla^2 U + U - U^3 - V + k$, $\dot V = (b \nabla^2 V + U - V) / \tau$.

**Coupling**: stage $k$'s normalised product modulates stage $k+1$'s feed rate ($F$ in GS) or resting bias ($k$ in FN), so spatial structure upstream drives where downstream patterns can form.

**Driving the patterns**:
1. `PreferenceDriver` — render a grid of variants, you pick one, the search distribution recenters on the chosen variant.
2. `PerceptualDriver` — a (1,$\lambda$) evolution strategy that maximises (or matches) a Gabor-bank perceptual distance to a target image.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # allow `import zebrastack` when running from examples/

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from zebrastack.rxndiff import (
    GrayScottStage, FitzHughNagumoStage,
    HierarchicalRDStack,
    PreferenceDriver, PerceptualDriver,
    perceptual_distance,
)

## 1. Single-stage sanity check

Run each kinetics in isolation and confirm we see characteristic patterns (spots / labyrinth for GS, branching stripes for FN).

In [ ]:
gs = GrayScottStage(size=128, F=0.0367, k=0.0649, seed=0)
gs.step(n_steps=4000)

fn = FitzHughNagumoStage(size=128, seed=0)
fn.step(n_steps=30000)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(gs.V, cmap='copper');     axes[0].set_title('Gray-Scott V');     axes[0].set_axis_off()
axes[1].imshow(fn.U, cmap='copper');     axes[1].set_title('FitzHugh-Nagumo U'); axes[1].set_axis_off()
plt.tight_layout(); plt.show()

## 2. Hierarchical Gray-Scott (two stages)

Stage 0 runs at one Gray-Scott parameter regime; stage 1's feed rate $F$ is locally boosted wherever stage 0 produced $V$. Compare the two outputs: stage 1 inherits the *layout* of stage 0 but expresses it in its own pattern grammar.

In [ ]:
stack = HierarchicalRDStack([
    GrayScottStage(size=128, F=0.0367, k=0.0649),         # spots/labyrinth
    GrayScottStage(size=128, F=0.030,  k=0.062, F_alpha=0.05),  # different regime, modulated
])
stack.reset(seed=42)
stack.step(n_steps=4000)

outs = stack.outputs()
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, img, title in zip(axes, outs, ['stage 0 (drives feed)', 'stage 1 (modulated)']):
    ax.imshow(img, cmap='copper'); ax.set_title(title); ax.set_axis_off()
plt.tight_layout(); plt.show()

## 3. Mixed cascade: Gray-Scott $\rightarrow$ FitzHugh-Nagumo

Different kinetics integrate at different time-scales. `substeps_per_stage` lets us run many FN inner steps per outer GS step, keeping each stage near its own stable $dt$.

In [ ]:
stack = HierarchicalRDStack([
    GrayScottStage(size=96, F=0.0367, k=0.0649),
    FitzHughNagumoStage(size=96, k_alpha=5e-3),
])
stack.reset(seed=1)
stack.step(n_steps=2000, substeps_per_stage=[1, 10])

outs = stack.outputs()
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(outs[0], cmap='copper'); axes[0].set_title('GS V (stage 0)'); axes[0].set_axis_off()
axes[1].imshow(outs[1], cmap='copper'); axes[1].set_title('FN V (stage 1, GS-modulated)'); axes[1].set_axis_off()
plt.tight_layout(); plt.show()

## 4. Spatial-scale modulation — three-stage cascade

Each stage takes a `scale` parameter that multiplies its diffusion coefficients by `scale**2` (so pattern wavelength $\lambda \propto \text{scale}$, since $\lambda \propto \sqrt{D}$). `dt` is auto-capped to satisfy the diffusion CFL limit, so larger-scale stages need proportionally more substeps to reach the same maturity. The seed-blob size scales with `scale` too so that nucleation works at every scale.

The three-stage stack below uses the same Gray-Scott $(F, k)$ regime everywhere; only the scale changes. Stage 0 (fine spots) modulates stage 1 (medium), which modulates stage 2 (coarse) — a fine-to-coarse hierarchy.

In [ ]:
from zebrastack.rxndiff import structure_score

scales = [1.0, 1.7, 2.8]
substeps = [1, 4, 10]
stack = HierarchicalRDStack([
    GrayScottStage(size=160, F=0.0367, k=0.0649, scale=scales[0]),
    GrayScottStage(size=160, F=0.0367, k=0.0649, scale=scales[1], F_alpha=0.05),
    GrayScottStage(size=160, F=0.0367, k=0.0649, scale=scales[2], F_alpha=0.06),
])
stack.reset(seed=0)
stack.step(n_steps=3500, substeps_per_stage=substeps)

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, img, s in zip(axes, stack.outputs(), stack.stages):
    ax.imshow(img, cmap='copper')
    ax.set_title(f'scale={s.scale:.1f}, dt={s.dt:.3f}\nstructure={structure_score(img):.2f}')
    ax.set_axis_off()
plt.tight_layout(); plt.show()

## 5. Preference-based driver (interactive breeder)

Render a $3\times3$ grid of variants (top-left is always the current centroid). You inspect the grid and call `driver.pick(children, idx)` with your favourite. The driver recenters its sampling distribution there and you re-run the cell.

In [ ]:
def make_stack():
    return HierarchicalRDStack([
        GrayScottStage(size=80, F=0.0367, k=0.0649),
        GrayScottStage(size=80, F=0.030,  k=0.062, F_alpha=0.05),
    ])

driver = PreferenceDriver(make_stack, n_variants=9, perturb=0.10, n_steps=2500)
rng = np.random.default_rng(0)

def render_grid(driver, rng):
    children, images = driver.generate(rng=rng)
    fig, axes = plt.subplots(3, 3, figsize=(7, 7))
    for i, (ax, img) in enumerate(zip(axes.flat, images)):
        ax.imshow(img, cmap='copper'); ax.set_axis_off()
        ax.set_title(f'#{i}' + (' (current)' if i == 0 else ''), fontsize=9)
    plt.tight_layout(); plt.show()
    return children

children = render_grid(driver, rng)

In [ ]:
# Pick your favourite (0..8) and re-render. Repeat until happy.
favourite = 4  # <-- edit this
driver.pick(children, favourite)
children = render_grid(driver, rng)

## 6. Perceptual driver — scoring modes

`PerceptualDriver` is a (1, λ) evolution strategy. The *scoring mode* decides what "best" means. A naïve "maximise distance from baseline" objective rewards drift, so the search often settles on near-degenerate fields that are merely *different*. The richer modes below rate candidates on their pattern content directly.

| `mode` | Score | Notes |
| --- | --- | --- |
| `'maximize'` | distance from a baseline in Gabor feature space | drifts; can land on near-uniform fields if those are far enough from the baseline |
| `'match'` | negated distance to a target image | useful when you have a reference texture |
| `'structure'` | multi-scale Gabor energy entropy × autocorrelation coherence (see `structure_score`) | rewards *multi-scale, organised* texture; explicitly rejects near-uniform outputs |
| `'novelty'` | mean distance to *k* nearest neighbours in an internal archive | each new accepted design joins the archive, so successive generations are pushed away from everything seen so far |
| `'novelty+structure'` | weighted sum of the two above | usually the best default — novelty for exploration, structure to keep results *interesting* rather than just different |

The cell below runs three modes from the same starting point and compares them side-by-side. Underneath, we look at the archive the `novelty+structure` run built up — a gallery of visually distinct patterns rather than a single "best".

In [ ]:
from zebrastack.rxndiff import structure_score

baseline_stack = make_stack(); baseline_stack.reset(seed=0); baseline_stack.step(n_steps=2000)
baseline = baseline_stack.output()

results = {}
for mode in ('maximize', 'structure', 'novelty+structure'):
    drv = PerceptualDriver(make_stack, mode=mode,
                           n_steps=2000, n_offspring=6, sigma=0.10)
    drv.run(n_generations=12, rng=np.random.default_rng(7))
    results[mode] = drv

fig, axes = plt.subplots(1, 4, figsize=(13, 3.6))
axes[0].imshow(baseline, cmap='copper')
axes[0].set_title(f'baseline\nstructure={structure_score(baseline):.2f}')
axes[0].set_axis_off()
for ax, (mode, drv) in zip(axes[1:], results.items()):
    ax.imshow(drv.best_image, cmap='copper')
    ax.set_title(f"{mode}\nbest={drv.best_score:.2f}, "
                 f"structure={structure_score(drv.best_image):.2f}")
    ax.set_axis_off()
plt.tight_layout(); plt.show()

In [ ]:
nov = PerceptualDriver(make_stack, mode='novelty+structure',
                       n_steps=2000, n_offspring=6, sigma=0.12,
                       archive_threshold=0.02)
nov.run(n_generations=20, rng=np.random.default_rng(13))

archive = nov.archive_images
n_show = min(8, len(archive))
cols = 4; rows = int(np.ceil(n_show / cols))
fig, axes = plt.subplots(rows, cols, figsize=(2.4 * cols, 2.4 * rows))
for ax, img in zip(np.atleast_1d(axes).flat, archive[:n_show]):
    ax.imshow(img, cmap='copper'); ax.set_axis_off()
for ax in np.atleast_1d(axes).flat[n_show:]:
    ax.set_axis_off()
fig.suptitle(f'Novelty archive ({len(archive)} accepted designs)')
plt.tight_layout(); plt.show()

### Match a target image

Set `mode='match'` and pass an arbitrary image (here a stylised checkerboard) as the target. The driver searches RD parameters whose product field has Gabor statistics closest to the target.

In [ ]:
yy, xx = np.mgrid[0:80, 0:80]
target = ((np.sin(xx * 0.5) * np.sin(yy * 0.5)) > 0).astype(float)

pdrv = PerceptualDriver(
    make_stack, target=target, mode='match',
    n_steps=2000, n_offspring=6, sigma=0.10,
)
pdrv.run(n_generations=10, rng=np.random.default_rng(2), verbose=True)
# 'match' score is negated distance (higher is better) — so distance to target is -best_score
match_distance = -pdrv.best_score

fig, axes = plt.subplots(1, 2, figsize=(7, 3.6))
axes[0].imshow(target,          cmap='copper'); axes[0].set_title('target');   axes[0].set_axis_off()
axes[1].imshow(pdrv.best_image, cmap='copper')
axes[1].set_title(f'best match (d={match_distance:.3f})'); axes[1].set_axis_off()
plt.tight_layout(); plt.show()